**Linda Zier**

**ST 554**

**HW #9**

## The Goal

• Finding a data set you can fit supervised learning models with

• Using a numeric or binary response, fitting three different classes of models and choosing an overall best model.

• Writing a narrative (via a notebook) with explanations and discussions as you go through the above.

## The Data

The dataset I used gives insights into bike rentals in Seoul Korea based on environmental factors.  I will use these as well as day of week and month considerations to model number of bikes rented.


**Setup**

First I have to setup everything. This includes installing pyspark, importing, and starting  my Spark Session.

In [3]:
# committing regularly
#!cd ST-554-repo && git add -A && git commit -m "progress" && git push

In [2]:
# clone my hub - just each first time I get started
#!git clone https://github.com/ljzier/ST-554-repo.git
!pip install pyspark


Defaulting to user installation because normal site-packages is not writeable


In [25]:

import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, SQLTransformer

from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GeneralizedLinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

print("imports ran")

imports ran


In [26]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [1]:
# read in the data and inspect
import pandas as pd

df = pd.read_csv('ST-554-repo/data/SeoulBikeData.csv', encoding='latin1')
print(df.head())
print(df.dtypes)
print(df.shape)


         Date  Rented Bike Count  Hour  Temperature(°C)  Humidity(%)  \
0  01/12/2017                254     0             -5.2           37   
1  01/12/2017                204     1             -5.5           38   
2  01/12/2017                173     2             -6.0           39   
3  01/12/2017                107     3             -6.2           40   
4  01/12/2017                 78     4             -6.0           36   

   Wind speed (m/s)  Visibility (10m)  Dew point temperature(°C)  \
0               2.2              2000                      -17.6   
1               0.8              2000                      -17.6   
2               1.0              2000                      -17.7   
3               0.9              2000                      -17.6   
4               2.3              2000                      -18.6   

   Solar Radiation (MJ/m2)  Rainfall(mm)  Snowfall (cm) Seasons     Holiday  \
0                      0.0           0.0            0.0  Winter  No Holiday   


I'll get an error if I try to run this again because I renamed my columns, FYI.  Only run it once at the start

In [27]:

#see if there are any non-functioning days
print(df['Functioning Day'].value_counts())
print(df[df['Functioning Day'] == 'No']['Rented Bike Count'].unique())

#filter out the non-functioning days and check the number of rows
df = df[df['Functioning Day'] == 'Yes']
print(df.shape)

KeyError: 'Functioning Day'

**Clean and Convert**

Since I already loaded my data, now I clean and convert my dataframe to a Sparkdata frame.

In [28]:


#rename columns to make the names spark-friendly (no spaces)
df.columns = [
    'Date', 'Bike_Count', 'Hour', 'Temperature', 'Humidity',
    'Wind_Speed', 'Visibility', 'Dew_Point', 'Solar_Radiation',
    'Rainfall', 'Snowfall', 'Seasons', 'Holiday', 'Functioning_Day'
]

#convert to Spark data frame
sdf= spark.createDataFrame(df)

# month as new feature
sdf = sdf.withColumn('Month', F.month(F.to_date(F.col('Date'), 'dd/MM/yyyy')))

# day of week as new feature
sdf = sdf.withColumn('Day_of_Week', F.dayofweek(F.to_date(F.col('Date'), 'dd/MM/yyyy')))

# make the type DoubleType which is like Float64
numeric_cols = ['Bike_Count', 'Hour', 'Temperature', 'Humidity', 'Wind_Speed', 
                'Visibility', 'Dew_Point', 'Solar_Radiation', 'Rainfall', 
                'Snowfall', 'Month', 'Day_of_Week']

for c in numeric_cols:
    sdf = sdf.withColumn(c, F.col(c).cast(DoubleType()))

sdf.printSchema()
sdf.show(5)

root
 |-- Date: string (nullable = true)
 |-- Bike_Count: double (nullable = true)
 |-- Hour: double (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- Visibility: double (nullable = true)
 |-- Dew_Point: double (nullable = true)
 |-- Solar_Radiation: double (nullable = true)
 |-- Rainfall: double (nullable = true)
 |-- Snowfall: double (nullable = true)
 |-- Seasons: string (nullable = true)
 |-- Holiday: string (nullable = true)
 |-- Functioning_Day: string (nullable = true)
 |-- Month: double (nullable = true)
 |-- Day_of_Week: double (nullable = true)

+----------+----------+----+-----------+--------+----------+----------+---------+---------------+--------+--------+-------+----------+---------------+-----+-----------+
|      Date|Bike_Count|Hour|Temperature|Humidity|Wind_Speed|Visibility|Dew_Point|Solar_Radiation|Rainfall|Snowfall|Seasons|   Holiday|Functioning_Day|Month|Day_of_Week|
+-

## Splitting the Data, Metrics, and Models

• Using spark MLlib, I split the data into a training and test set with an 80/20 split.

• I've chosen to use RMSE as the metric to judge my models.

• The three different classes of models I'll be fitting are: 

    1.Linear Regression with Elastic Net:  Elastic Net linear regression is a regularized method that combines L1 (Lasso) and L2 (Ridge) penalties to overcome limitations of both. It performs feature selection (driving some coefficients to zero) while keeping correlated predictors together. This works well when multiple features (variables) are correlated. Since environmental factors like temperature and humidity are often correlated, I thtink this is a promising model.
    
    2.Random Forest Regressor: Random forest is an ML algorithm that combines the output of multiple decision trees to reach a single result, crreating a more stable and accurate model.  It uses bootstrapping: Each tree is trained on a random sample of the training data, allowing for different data subsets. Prediction: For classification, the model uses majority voting; for regression, it averages predictions.
    
    3. Generalized Linear Regressor (Poisson): Generalized Linear Regression (Poisson): Regular linear regression assumes the response variable is continuous and normally distributed. But Rented Bike Count is a count that is always a whole number and can't be negative. Count data often follows a Poisson distribution instead of a normal one. Poisson regression handles this by modeling the log of the expected count as a linear combination of the predictors, which also guarantees predictions are always positive.

In [29]:
# split into training and test roughly 80/20
train, test = sdf.randomSplit([0.8, 0.2], seed = 8)



Creating my pipeline transformations.

In [33]:
# StringIndexer to convert seasons and holidays to numeric
indexer = StringIndexer(
    inputCols=['Seasons', 'Holiday'],
    outputCols=['Seasons_idx', 'Holiday_idx'])

# OneHotEncoder to convert index to dummy
encoder =OneHotEncoder(
    inputCols=['Seasons_idx', 'Holiday_idx'],
    outputCols=['Seasons_enc', 'Holiday_enc'])

# SQLTransformer to log transform response
sqlTrans= SQLTransformer(statement = 
                         "SELECT *, log(Bike_Count) as label FROM __THIS__")

#VectorAssembler to bundle features together into one vector
assembler = VectorAssembler(
    inputCols=['Hour', 'Temperature', 'Humidity', 'Wind_Speed', 'Visibility',
               'Dew_Point', 'Solar_Radiation', 'Rainfall', 'Snowfall',
               'Month', 'Day_of_Week', 'Seasons_enc', 'Holiday_enc'],
    outputCol='features')

print("transformations complete")

transformations complete


## Model Fitting
Use Spark MLlib to fit your three different classes models to the training data. This
should be done using pipelines and cross validation to choose your best model for each model type. You
should compare your models using your metric chosen earlier.

• You should set up a pipeline in pyspark for each of your models

• You should do your transformations using the functions from MLlib to easily put them into the pipeline.

### Linear Regression Model with elastic net

In [34]:
from pyspark.ml import Pipeline

# define lr model
lr = LinearRegression()

paramGrid_lr = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.01, 0.05, 0.1]) \
    .addGrid(lr.elasticNetParam, [0, 0.5, 1.0]) \
    .build()

#build lr pipeline
pipeline_lr= Pipeline(stages = [indexer, encoder, sqlTrans, assembler, lr])

crossval_lr = CrossValidator(estimator = pipeline_lr,
                          estimatorParamMaps = paramGrid_lr,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)
#fit trraining data
cvModel_lr = crossval_lr.fit(train)
print("LR done")

#cvModel_lr.transform(test) #for predictions

26/04/13 17:33:56 WARN CacheManager: Asked to cache already cached data.
26/04/13 17:33:56 WARN CacheManager: Asked to cache already cached data.
26/04/13 17:33:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/13 17:33:57 WARN Instrumentation: [db2a510a] regParam is zero, which might cause numerical instability and overfitting.
26/04/13 17:34:00 WARN Instrumentation: [47f3fc86] regParam is zero, which might cause numerical instability and overfitting.
26/04/13 17:34:01 WARN Instrumentation: [101b5346] regParam is zero, which might cause numerical instability and overfitting.
26/04/13 17:34:11 WARN Instrumentation: [c53f327b] regParam is zero, which might cause numerical instability and overfitting.
26/04/13 17:34:12 WARN Instrumentation: [284966d5] regParam is zero, which might cause numerical instability and overfitting.
26/04/13 17:34:13 WARN Instrumentat

LR done


Now we'll check which parameters were best. From below it is regParm=0, elasticNetParm=0 with an RMSE=0.713. Including all the coefficients with no shrinking was best.

In [35]:
#using professor's code 

my_list = []
for i in range(len(paramGrid_lr)):
    my_list.append([cvModel_lr.avgMetrics[i], paramGrid_lr[i].values()])
my_list

[[0.7132493175005555, dict_values([0.0, 0.0])],
 [0.7132493175005551, dict_values([0.0, 0.5])],
 [0.7132493175005552, dict_values([0.0, 1.0])],
 [0.7152517705944031, dict_values([0.01, 0.0])],
 [0.7158404671703302, dict_values([0.01, 0.5])],
 [0.7174036475382495, dict_values([0.01, 1.0])],
 [0.71830994212836, dict_values([0.05, 0.0])],
 [0.7259186464007465, dict_values([0.05, 0.5])],
 [0.7339100323779431, dict_values([0.05, 1.0])],
 [0.7213993843918304, dict_values([0.1, 0.0])],
 [0.7375043950487381, dict_values([0.1, 0.5])],
 [0.7600825210967337, dict_values([0.1, 1.0])]]

### Random Forest Model

In [37]:
## Defining random forest model
rf = RandomForestRegressor(seed=8)

paramGrid_rf = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 20]) \
    .addGrid(rf.maxDepth, [3, 5]) \
    .build()
    
# rf pipeline 
pipeline_rf= Pipeline(stages = [indexer, encoder, sqlTrans, assembler, rf])


crossval_rf = CrossValidator(estimator = pipeline_rf,
                          estimatorParamMaps = paramGrid_rf,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

#fit trraining data
cvModel_rf = crossval_rf.fit(train)

print("RF done")

26/04/13 17:55:51 WARN DAGScheduler: Broadcasting large task binary with size 1370.2 KiB
26/04/13 17:56:08 ERROR Executor: Exception in task 91.0 in stage 1004.0 (TID 42978)
java.lang.OutOfMemoryError: Java heap space
26/04/13 17:56:08 ERROR Executor: Exception in task 67.0 in stage 1004.0 (TID 42954)
java.lang.OutOfMemoryError: Java heap space
26/04/13 17:56:10 ERROR Executor: Exception in task 100.0 in stage 1004.0 (TID 42987)
java.lang.OutOfMemoryError: Java heap space
26/04/13 17:56:11 ERROR Executor: Exception in task 34.0 in stage 1004.0 (TID 42921)
java.lang.OutOfMemoryError: Java heap space
26/04/13 17:56:10 ERROR Executor: Exception in task 12.0 in stage 1004.0 (TID 42899)
java.lang.OutOfMemoryError: Java heap space
	at java.base/java.lang.Integer.valueOf(Integer.java:1081)
	at scala.runtime.java8.JFunction2$mcIII$sp.apply(JFunction2$mcIII$sp.scala:17)
	at scala.collection.ArrayOps$.scanLeft$extension(ArrayOps.scala:824)
	at org.apache.spark.ml.tree.impl.DTStatsAggregator.<ini

Py4JJavaError: An error occurred while calling o37604.fit.
: org.apache.spark.SparkException: Job 509 cancelled because SparkContext was shut down
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1(DAGScheduler.scala:1301)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1$adapted(DAGScheduler.scala:1299)
	at scala.collection.mutable.HashSet$Node.foreach(HashSet.scala:450)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:376)
	at org.apache.spark.scheduler.DAGScheduler.cleanUpAfterSchedulerStop(DAGScheduler.scala:1299)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onStop(DAGScheduler.scala:3234)
	at org.apache.spark.util.EventLoop.stop(EventLoop.scala:85)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$stop$3(DAGScheduler.scala:3120)
	at org.apache.spark.util.Utils$.tryLogNonFatalError(Utils.scala:1300)
	at org.apache.spark.scheduler.DAGScheduler.stop(DAGScheduler.scala:3120)
	at org.apache.spark.S

26/04/13 17:57:53 ERROR TaskContextImpl: Error in TaskCompletionListener
org.apache.spark.SparkException: Block broadcast_723 does not exist
	at org.apache.spark.errors.SparkCoreErrors$.blockDoesNotExistError(SparkCoreErrors.scala:322)
	at org.apache.spark.storage.BlockInfoManager.blockInfo(BlockInfoManager.scala:269)
	at org.apache.spark.storage.BlockInfoManager.unlock(BlockInfoManager.scala:390)
	at org.apache.spark.storage.BlockManager.releaseLock(BlockManager.scala:1344)
	at org.apache.spark.broadcast.TorrentBroadcast.$anonfun$releaseBlockManagerLock$1(TorrentBroadcast.scala:323)
	at org.apache.spark.broadcast.TorrentBroadcast.$anonfun$releaseBlockManagerLock$1$adapted(TorrentBroadcast.scala:323)
	at org.apache.spark.TaskContext$$anon$1.onTaskCompletion(TaskContext.scala:137)
	at org.apache.spark.TaskContextImpl.$anonfun$invokeTaskCompletionListeners$1(TaskContextImpl.scala:153)
	at org.apache.spark.TaskContextImpl.$anonfun$invokeTaskCompletionListeners$1$adapted(TaskContextImpl.sc

### Generalized Linear Regression Poisson Model

## Model Testing
Lastly, you should evaluate the best models from each class on the test set and state which overall model
was deemed the best.
